# Görev 4: Self-Attention Head'in Bigram Modeline Entegrasyonu ve Eğitimi

### Amaç:
1. `Head` sınıfını `nn.Module` olarak tanımlamak (Q, K, V projeksiyonları, $\sqrt{d_k}$ ölçekleme, nedensel maskeleme ve softmax).
2. Bu Head'i ve pozisyon embedding'ini `BigramLanguageModel` içine yerleştirmek.
3. Modeli eğitmek ve nihai validation loss'u Görev 1'deki Bigram tabanıyla yan yana kıyaslamak.
4. Kaybın ne kadar ve neden düştüğünü kendi cümlelerimizle izah etmek.
5. Metin üretirken girdiyi neden son `block_size` harfe kırpmak zorunda olduğumuzu açıklamak.

> **Referans:** Andrej Karpathy - "Let's build GPT: from scratch, in code, spelled out" (Video: 1:19:11 - 1:21:59)

### 1. Veri Hazırlığı ve Ortam Kurulumu

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import pandas as pd

In [2]:
batch_size = 32
block_size = 8
max_iters = 3000
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32
torch.manual_seed(42)

In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [6]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [7]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


In [8]:
print(f"Cihaz: {device}, Vocab Size: {vocab_size}, n_embd: {n_embd}")

Cihaz: cuda, Vocab Size: 65, n_embd: 32


### Self-Attention Head Sınıfı

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # tril değişkeni model parametresi değil sabit bir tensördür
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        # Attention score
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5) # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)

        # ağırlıklı toplam
        v = self.value(x) # (B, T, head_size)
        out = wei @ v     # (B, T, head_size)
        return out

### 3. Self-Attention Head Eklenmiş Bigram Modeli

In [ ]:
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # self-attention head
        self.sa_head = Head(n_embd)
        # Son karaktere karar veren katman
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, n_embd)
        x = tok_emb + pos_emb # (B, T, n_embd)

        # Self-attention uygula
        x = self.sa_head(x) # (B, T, n_embd)

        # Logitleri üret
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx: (B, T) boyutunda mevcut context
        for _ in range(max_new_tokens):
            # KRİTİK ADIM: Pozisyon embedding sınırını aşmamak için girdiyi son block_size karaktere kırpıyoruz!
            idx_cond = idx[:, -block_size:]
            # tahminleri al
            logits, loss = self(idx_cond)
            # sadece son zaman adımına odaklan
            logits = logits[:, -1, :] # (B, C)
            # olasılıkları hesapla
            probs = F.softmax(logits, dim=-1) # (B, C)
            # dağılımdan örnekle
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # yeni token'ı bağlama ekle
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

### 4. Modelin Eğitilmesi

In [4]:
torch.manual_seed(1337)
model = BigramLanguageModel().to(device)
print(f"Toplam Parametre Sayısı: {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Eğitim Başlıyor...")
for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        print(f"step {iter:4d}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

Toplam Parametre Sayısı: 7,553


Eğitim Başlıyor...


step    0: train loss 4.2000, val loss 4.2047


step  300: train loss 2.9418, val loss 2.9574


step  600: train loss 2.6309, val loss 2.6456


...

step 1800: train loss 2.4520, val loss 2.4641


step 2100: train loss 2.4385, val loss 2.4422


step 2400: train loss 2.4241, val loss 2.4419


step 2700: train loss 2.4122, val loss 2.4340


step 2999: train loss 2.4230, val loss 2.4245


### 5. Metin Üretimi ve Sonuçların Alınması

In [5]:
final_losses = estimate_loss(model)
task4_train_loss = final_losses['train']
task4_val_loss = final_losses['val']

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_text = decode(model.generate(context, max_new_tokens=500)[0].tolist())

print("--- Self-Attention Head Modeli Tarafından Üretilen Metin ---")
print(generated_text)

--- Self-Attention Head Modeli Tarafından Üretilen Metin ---

And; bef bridcowf,
This by bth

Hiset bobe ale.
S:
O-' sthealilanss ar hthar uwqort vet?
F dilasoate awice my.

HDET:
ANGo our
Yowns, tof isth bot mil; dilll,
Wh iree sen cin lat Het drov ts, ane th poran ilerans!
el lind peal.
-hul wonchiry ptupr aiss hiwhy.
Hllines norodpeeelaves
Momy ys, dl tthake ont-wou when eiibyo wouti douris hane, ire st sousowrixs bure kad ntert the son; igr the:
EN CI inleront ffaf Pre?

Whiom.

He-NLIERLKENSby ake adsal thes ghest hoin cour ayraney Iry ts I fr af ve y


### 6. Karşılaştırma Tablosu: Görev 1 Tabanı ile Yan Yana

In [6]:
# Görev 1'deki Bigram taban modelinin değerleri
task1_val_loss = 2.5135 # Görev 1 baseline validation loss
diff = task1_val_loss - task4_val_loss

df_comparison = pd.DataFrame({
    "Model Mimarisi": [
        "Görev 1: Saf Bigram Tabanı (Lookup Table)",
        "Görev 4: Bigram + Pozisyon Emb + 1x Self-Attention Head"
    ],
    "Val Loss": [
        f"{task1_val_loss:.4f}",
        f"{task4_val_loss:.4f}"
    ],
    "Fark (Düşüş Miktarı)": [
        "Referans (Taban)",
        f"-{diff:.4f} puan iyileşme"
    ],
    "Yorum": [
        "Token'lar arası iletişim yok, sadece 1 önceki karaktere bakıyor",
        "Geçmiş 8 karaktere bakabiliyor ve veri odaklı dikkat uyguluyor"
    ]
})

print("="*80)
print("GÖREV 1 vs GÖREV 4 KARŞILAŞTIRMA TABLOSU")
print("="*80)
print(df_comparison.to_string(index=False))
print("="*80)

GÖREV 1 vs GÖREV 4 KARŞILAŞTIRMA TABLOSU
                                         Model Mimarisi Val Loss  Fark (Düşüş Miktarı)                                                           Yorum
              Görev 1: Saf Bigram Tabanı (Lookup Table)   2.5135      Referans (Taban) Token'lar arası iletişim yok, sadece 1 önceki karaktere bakıyor
Görev 4: Bigram + Pozisyon Emb + 1x Self-Attention Head   2.4251 -0.0884 puan iyileşme  Geçmiş 8 karaktere bakabiliyor ve veri odaklı dikkat uyguluyor


### 7. Değerlendirme ve Soruların Cevapları (Kendi Cümlelerimizle)

---

#### 1. Soru: Kayıp ne kadar düştü, neden düştü?
- **Ne kadar düştü?**:
  Validation loss, Görev 1'deki **~2.51** seviyesinden **~2.41 - 2.42** seviyesine geriledi (yaklaşık **0.10 puanlık** net ve belirgin bir düşüş).
- **Neden düştü?**:
  1. **Geçmiş Bağlamı Görebilme (Context Window):** Görev 1'deki saf Bigram modeli yalnızca tek bir önceki karaktere bakabiliyordu. Önceki harflerin hiçbiri dikkate alınmıyordu. Oysa bir dilde doğru harfi tahmin edebilmek için kelimenin ve cümlenin geçmişine ihtiyaç vardır.
  2. **Veri-Odaklı İletişim (Communication via Self-Attention):** Eklediğimiz Self-Attention head sayesinde, dizideki her token kendisinden önceki 8 karakterin tamamıyla iletişim kurabilir hale geldi. Üstelik bu iletişim sıradan bir ortalama değil; Query ve Key matrislerinin öğrendiği ağırlıklarla, o anki tahmin için en kritik karakterlere odaklanan dinamik bir iletişimdir.
  3. **Pozisyon Bilgisi (Position Embedding):** Token'ların cümle içindeki sırası modele tanıtıldı; böylece model karakterlerin sadece varlığını değil, geliş sırasını da öğrendi. Bu faktörler modelin tahmin gücünü doğrudan artırarak kaybı düşürdü.

---

#### 2. Soru: Metin üretirken girdiyi neden son `block_size` harfe kırpmak zorundayız (`idx[:, -block_size:]`)?
1. **Pozisyon Embedding Sınırı (Out of Bounds / IndexError):**
   Modelimizde pozisyon embedding tablosu `self.position_embedding_table = nn.Embedding(block_size, n_embd)` olarak tanımlanmıştır. 
   Metin üretimi sırasında (`generate`), her adımda bir harf üretilip `idx` tensörünün sonuna eklenir ($T = 1, 2, 3, \dots, 500$). 
   Eğer girdi kırpılmazsa, $T > \text{block\_size}$ (örneğin $T=9, 10, \dots$) olduğunda `torch.arange(T)` çağrısı embedding tablosunun sınırlarını aşan indeksler üretir ve PyTorch **`IndexError: index out of range in self`** hatası verir.
2. **Nedensel Maske (tril) Boyutu:**
   `Head` modülümüzün içine kaydettiğimiz `tril` maskesi de `(block_size, block_size)` boyutundadır. $T > \text{block\_size}$ olduğunda `self.tril[:T, :T]` dilimlemesi boyut aşımı hatasına yol açacaktır.
3. **Modelin Eğitildiği Bağlam Kapasitesi:**
   Modelimiz en fazla `block_size` (8) karakterlik geçmişe bakacak şekilde tasarlanmış ve eğitilmiştir. Bu yüzden 500 karakterlik bir metin üretilirken dahi, en son üretilen harfin tahmini için yalnızca en güncel son 8 harfin (`idx[:, -block_size:]`) modele girdi olarak verilmesi mimari bir zorunluluktur.